In [26]:
import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow.keras.layers import Dense, Dropout, Input, Concatenate
from tensorflow.keras.models import Model
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.model_selection import train_test_split
import joblib

In [17]:
n = 2000

In [31]:
np.random.seed(0)

In [18]:
df = pd.DataFrame({
    'exp': np.random.randint(0, 25, n),
    'work_hpw': np.random.normal(45, 8, n).clip(20, 70),
    'score': np.random.normal(6.5, 1.5, n).clip(1, 10),
    'certif': np.random.poisson(1.5, n),
    'edu': np.random.choice(['High School', 'Bachelor', 'Master', 'PhD'], n, p=[0.15, 0.45, 0.3, 0.1]),
    'city': np.random.choice(['Tier1', 'Tier2', 'Tier3'], n, p=[0.4, 0.35, 0.25]),
    'role': np.random.choice(['Data Analyst', 'Software Engineer', 'Manager', 'Sales', 'HR'], n),
    'size': np.random.choice(['Small', 'Medium', 'Large'], n, p=[0.3, 0.45, 0.25]),
})

In [19]:
df['salary'] = (
    500
    + 45 * df['exp']
    + 6 * df['work_hpw']
    + 120 * df['score']
    + 80 * df['certif']
    + 250 * df['edu'].map({'High School': 0, 'Bachelor': 1, 'Master': 2, 'PhD': 3})
    + 200 * df['city'].map({'Tier1': 2, 'Tier2': 1, 'Tier3': 0})
    + df['role'].map({'Data Analyst': 800, 'Software Engineer': 1200, 'Manager': 1500, 'Sales': 700, 'HR': 650})
    + df['size'].map({'Small': 0, 'Medium': 300, 'Large': 700})
    + np.random.normal(0, 250, n)
).round(2)

In [20]:
promotion_prob = (
    1 / (1 + np.exp(-(
        0.15 * df['exp']
        + 0.6 * df['score']
        + 0.4 * df['certif']
        + 0.5 * df['edu'].map({'High School': 0, 'Bachelor': 1, 'Master': 2, 'PhD': 3})
        - 5.5
    )))
)
df['promotion'] = (promotion_prob > 0.5).astype(int)

In [21]:
X = df.drop(['salary', 'promotion'], axis = 1)
y_reg = df['salary']
y_clf = df['promotion']

In [23]:
num_fts = ['exp', 'work_hpw', 'score', 'certif']
nom_fts = ['role']
ord_fts = ['edu', 'city', 'size']

In [25]:
ord_ords = [['High School', 'Bachelor', 'Master', 'PhD'],
            ['Tier3', 'Tier2', 'Tier1'],
            ['Small', 'Medium', 'Large']]

In [27]:
num_pp = Pipeline([
    ('imputer', SimpleImputer(strategy = 'mean')),
    ('scaler', StandardScaler())
])

In [28]:
nom_pp = Pipeline([
    ('onehot', OneHotEncoder(sparse_output = False))
])

In [29]:
ord_pp = Pipeline([
    ('encoder', OrdinalEncoder(categories = ord_ords))
])

In [30]:
pp = ColumnTransformer([
    ('num', num_pp, num_fts),
    ('nom', nom_pp, nom_fts),
    ('ord', ord_pp, ord_fts)
])

In [32]:
x_train, x_test, y_reg_train, y_reg_test, y_clf_train, y_clf_test = train_test_split(
    X, y_reg, y_clf, test_size = 0.25, random_state = 0)

In [33]:
x_train = pp.fit_transform(x_train)
x_test = pp.transform(x_test)

In [41]:
inputs = Input(shape = (x_train.shape[1], ))

A1 = Dense(8, activation = 'relu')(inputs)
A2 = Dense(4, activation = 'relu')(A1)
A3 = Dense(16, activation = 'relu')(A2)
A4 = Dropout(0.2)(A3)

R1 = Dense(16, activation = 'relu')(A4)

C1 = Dense(8, activation = 'relu')(A4)

fused = Concatenate()([R1, C1])

reg_out = Dense(1, activation = 'linear', name = 'reg_out')(fused)
clf_out = Dense(1, activation = 'sigmoid', name = 'clf_out')(fused)

In [42]:
model = Model(inputs = inputs, outputs = [reg_out, clf_out])

In [43]:
model.summary()

Model: "functional_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)                  ┃ Output Shape              ┃         Param # ┃ Connected to               ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│ input_layer_5 (InputLayer)    │ (None, 12)                │               0 │ -                          │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_14 (Dense)              │ (None, 8)                 │             104 │ input_layer_5[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_15 (Dense)              │ (None, 4)                 │              36 │ dense_14[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_16 (Dense)              │ (None, 16)                │              80 │ dense_15[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dropout_2 (Dropout)           │ (None, 16)                │               0 │ dense_16[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_17 (Dense)              │ (None, 16)                │             272 │ dropout_2[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ dense_18 (Dense)              │ (None, 8)                 │             136 │ dropout_2[0][0]            │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ concatenate_2 (Concatenate)   │ (None, 24)                │               0 │ dense_17[0][0],            │
│                               │                           │                 │ dense_18[0][0]             │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ reg_out (Dense)               │ (None, 1)                 │              25 │ concatenate_2[0][0]        │
├───────────────────────────────┼───────────────────────────┼─────────────────┼────────────────────────────┤
│ clf_out (Dense)               │ (None, 1)                 │              25 │ concatenate_2[0][0]        │
└───────────────────────────────┴───────────────────────────┴─────────────────┴────────────────────────────┘

 Total params: 678 (2.65 KB)

 Trainable params: 678 (2.65 KB)

 Non-trainable params: 0 (0.00 B)

In [49]:
model.compile(
    optimizer = 'adam',
    loss = {'reg_out': 'mae', 'clf_out': 'binary_crossentropy'},
    metrics = {'reg_out': ['mse'], 'clf_out': ['accuracy']}
)

In [50]:
model.fit(
    x_train, {'reg_out': y_reg_train, 'clf_out': y_clf_train},
    epochs = 100,
    batch_size = 64,
    validation_split = 0.1,
    verbose = 1
)

Epoch 1/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 9s 62ms/step - clf_out_accuracy: 0.7185 - clf_out_loss: 0.6790 - loss: 4036.0801 - reg_out_loss: 4033.2705 - reg_out_mse: 16726498.0000 - val_clf_out_accuracy: 0.7733 - val_clf_out_loss: 0.6653 - val_loss: 3979.1292 - val_reg_out_loss: 4003.8545 - val_reg_out_mse: 16248028.0000
Epoch 2/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 15ms/step - clf_out_accuracy: 0.7896 - clf_out_loss: 0.6454 - loss: 4035.7366 - reg_out_loss: 4018.4971 - reg_out_mse: 16724000.0000 - val_clf_out_accuracy: 0.8467 - val_clf_out_loss: 0.6144 - val_loss: 3978.6072 - val_reg_out_loss: 4003.3750 - val_reg_out_mse: 16244284.0000
Epoch 3/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 13ms/step - clf_out_accuracy: 0.8059 - clf_out_loss: 0.5770 - loss: 4034.9836 - reg_out_loss: 4040.9202 - reg_out_mse: 16718452.0000 - val_clf_out_accuracy: 0.8467 - val_clf_out_loss: 0.5180 - val_loss: 3977.4624 - val_reg_out_loss: 4002.3191 - val_reg_out_mse: 16235955.0000
Epoch 4/100
22/22 ━━━━━━━━━━━━━━━━━━━━ 0s 16ms

In [52]:
results = model.evaluate(x_test, {'reg_out': y_reg_test, 'clf_out': y_clf_test}, verbose = 0)

In [53]:
print(results)

[234.1381072998047, 233.17538452148438, 0.6236467957496643, 0.7839999794960022, 88764.5078125]


In [55]:
joblib.dump(model, 'model.joblib')
joblib.dump(pp, 'pp.joblib')

['pp.joblib']